# Pruning-only QAT frontier

This notebook starts from the selected mixed-precision QAT checkpoint and explores additional sparse weight compression. It contains no knowledge distillation. Every 30–50% candidate receives supervised recovery, masks are enforced after every optimizer step, and selection uses validation accuracy plus actual QPK2 file bytes. The CIFAR-10 test set is evaluated once, after validation selection.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2
from pathlib import Path
QAT_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/qat-mixed/qat-mp-w4dw6edgew8-a6-seed6886-best-target.pt'
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'
!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints results/logs experiments/pruning
!cp $QAT_SOURCE results/checkpoints/qat-mp-w4dw6edgew8-a6-seed6886-best-target.pt
!sha256sum results/checkpoints/qat-mp-w4dw6edgew8-a6-seed6886-best-target.pt
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
required = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta']
assert all((cifar_dir / name).is_file() for name in required), f'Missing CIFAR-10 files under {cifar_dir}'
print('Train: crop/flip; validation/test: normalization only; seed-6886 45k/5k split.')

In [ ]:
!set -o pipefail; python -m unittest discover -s tests -v 2>&1 | tee results/logs/pruning-correctness.log
!set -o pipefail; nvidia-smi 2>&1 | tee results/logs/pruning-gpu.log

In [ ]:
RUN_NAME = 'pruning-frontier-mp-w4dw6edgew8-a6-seed6886'
QAT_CHECKPOINT = 'results/checkpoints/qat-mp-w4dw6edgew8-a6-seed6886-best-target.pt'
SPARSITIES = '0.30 0.35 0.40 0.45 0.50'
RECOVERY_EPOCHS = 10
PRUNING_WARMUP_EPOCHS = 4
MAX_VALIDATION_DROP = 0.50
assert Path(QAT_CHECKPOINT).is_file()

In [ ]:
!set -o pipefail; python -m src.prune_qat --checkpoint {QAT_CHECKPOINT} --data-dir "{DATA_DIR}" --device cuda --epochs {RECOVERY_EPOCHS} --pruning-warmup-epochs {PRUNING_WARMUP_EPOCHS} --initial-sparsity 0.30 --sparsities {SPARSITIES} --max-validation-drop {MAX_VALIDATION_DROP} --run-name {RUN_NAME} 2>&1 | tee results/logs/{RUN_NAME}.log

In [ ]:
import json
run_dir = Path('experiments/pruning') / RUN_NAME
results = json.loads((run_dir / 'results.json').read_text())
winner = results['validation_selected']
rows = [{'sparsity': row['sparsity'], 'val_accuracy': row['best_validation_accuracy'], 'val_drop_pp': row['validation_drop'], 'MiB': row['total_bytes'] / 2**20, 'weight_ratio': row['weight_compression_ratio'], 'effective_zero_fraction': 1 - row['stored_nonzero_values'] / row['eligible_weight_values']} for row in results['candidates']]
display(rows)
print(results['selection_rule'])
print(json.dumps(winner, indent=2))
assert Path(winner['artifact']).stat().st_size == winner['total_bytes']

## Final held-out evaluation

Run exactly once after the code above has selected the winner. Do not use this result to revisit the sparsity choice.

In [ ]:
WINNER_CHECKPOINT = winner['checkpoint']
!set -o pipefail; python -m src.evaluate --checkpoint {WINNER_CHECKPOINT} --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/{RUN_NAME}-held-out-test.log

In [ ]:
import hashlib, tarfile
from IPython.display import FileLink
paths = [run_dir, Path('results/logs/pruning-correctness.log'), Path('results/logs/pruning-gpu.log'), Path('results/logs') / f'{RUN_NAME}.log', Path('results/logs') / f'{RUN_NAME}-held-out-test.log']
paths = [path for path in paths if path.exists()]
for path in paths:
    if path.is_file(): print(f'{hashlib.sha256(path.read_bytes()).hexdigest()}  {path}')
archive = Path(f'{RUN_NAME}-artifacts.tgz')
with tarfile.open(archive, 'w:gz') as tar:
    for path in paths: tar.add(path, arcname=str(path))
print(f'Created {archive} ({archive.stat().st_size:,} bytes)')
FileLink(str(archive))